# Feature engineering

Our Jupyter notebook, Error_Analysis ran a error and fairness analysis on our Neural Network and LightGBM model. Creating a threshold we discovered that both models comparitivly performed the same with LightGBM having 50 false alarms. The most profound discovery in our error analysis was that our models were struggling to capture signal for sudden applicant behavioral changes, for example, some applicants may have been good at paying back loans or had a good credit history but suddenly found themselves struggling to pay back debt. 

In [1]:
import sqlite3
import time 
from pathlib import Path

import numpy as np
import pandas as pd

DB_PATH = Path(r"C:\Data_Projects\Project_2_Data4401\home_credit.db")
PROCESSED = Path(r"C:\Data_Projects\Project_2_Data4401\data\processed")

conn = sqlite3.connect(DB_PATH)
# Setting up 
conn.execute("PRAGMA mmap_size = 4000000000")
conn.execute("PRAGMA cache_size = -2000000")
conn.execute("PRAGMA temp_store = MEMORY")

# Current Application

In [2]:
def run_query(name, sql):
    start = time.time()
    df = pd.read_sql(sql, conn)
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns in {time.time() - start:.0f}s")
    return df

In [3]:
BASE_DATA = """
SELECT *
FROM v_application_full
""" # everything merged with each other already
df_base = run_query("base", BASE_DATA)

base: 356,255 rows x 123 columns in 16s


In [4]:
df_base["NO_EMPLOYER"] = (df_base["DAYS_EMPLOYED"] == 365243).astype(int) #365243 is a placeholder also makes binary
df_base["DAYS_EMPLOYED"] = (df_base["DAYS_EMPLOYED"].replace(365243, np.nan))

print(df_base["dataset"].value_counts())
print(f"Clients with no TARGET (should equal the test count): {df_base['TARGET'].isna().sum():,}")


dataset
train    307511
test      48744
Name: count, dtype: int64
Clients with no TARGET (should equal the test count): 48,744


# Applicant History
### Work Being Done:
1. The client's past: other lenders and earlier Home Credit Loans
2. The 5 history tables (burea & balance, previous_application, pos_cash_balance, credit_card_balance, installments_payments)
3. **Aggregated** to one client 

| Source table | Rows before | Summary features |
|---|---:|---:|
| `bureau` + `bureau_balance` | 1.7M + 27.3M | 15 |
| `previous_application` | 1.7M | 11 |
| `pos_cash_balance` | 10.0M | 5 |
| `credit_card_balance` | 3.8M | 9 |
| `installments_payments` | 13.6M | 7 |
| **Result** | **~57M** | **356,255 applicants × 46 features** |


In [5]:
HISTORY_SQL = """
WITH bb AS (
    SELECT SK_ID_BUREAU,
        COUNT(*) AS bb_months,
        AVG(STATUS IN ('1','2','3','4','5')) AS bb_pct_late,
        MAX(CASE WHEN STATUS IN ('1','2','3','4','5') THEN CAST(STATUS AS INT) ELSE 0 END) AS bb_worst_status
    FROM bureau_balance
    GROUP BY SK_ID_BUREAU
),

bureau_agg AS (
    SELECT b.SK_ID_CURR,
        COUNT(*) AS bureau_n_loans,
        SUM(b.CREDIT_ACTIVE = 'Active') AS bureau_n_active,
        SUM(b.CREDIT_ACTIVE = 'Closed') AS bureau_n_closed,
        SUM(b.AMT_CREDIT_SUM) AS bureau_total_credit,
        SUM(b.AMT_CREDIT_SUM_DEBT) AS bureau_total_debt,
        SUM(b.AMT_CREDIT_SUM_OVERDUE) AS bureau_total_overdue,
        MAX(b.AMT_CREDIT_MAX_OVERDUE) AS bureau_max_overdue,
        SUM(b.AMT_CREDIT_SUM_DEBT) / NULLIF(SUM(b.AMT_CREDIT_SUM), 0) AS bureau_debt_ratio,
        MIN(b.DAYS_CREDIT) AS bureau_oldest_loan_days,
        MAX(b.DAYS_CREDIT) AS bureau_newest_loan_days,
        MAX(b.CREDIT_DAY_OVERDUE) AS bureau_max_days_overdue,
        SUM(b.CNT_CREDIT_PROLONG) AS bureau_n_prolonged,
        AVG(bb.bb_pct_late) AS bureau_avg_pct_months_late,
        MAX(bb.bb_worst_status) AS bureau_worst_status
    FROM bureau AS b
    LEFT JOIN bb ON b.SK_ID_BUREAU = bb.SK_ID_BUREAU
    GROUP BY b.SK_ID_CURR
),

prev_agg AS (
    SELECT SK_ID_CURR,
        COUNT(*) AS prev_n_apps,
        SUM(NAME_CONTRACT_STATUS = 'Approved') AS prev_n_approved,
        SUM(NAME_CONTRACT_STATUS = 'Refused') AS prev_n_refused,
        AVG(NAME_CONTRACT_STATUS = 'Refused') AS prev_pct_refused,
        AVG(AMT_APPLICATION) AS prev_avg_amt_applied,
        AVG(AMT_CREDIT) AS prev_avg_amt_credit,
        AVG(AMT_CREDIT / NULLIF(AMT_APPLICATION, 0)) AS prev_avg_credit_to_applied,
        AVG(AMT_ANNUITY) AS prev_avg_annuity,
        AVG(CNT_PAYMENT) AS prev_avg_n_payments,
        MAX(DAYS_DECISION) AS prev_last_decision_days,
        AVG(RATE_DOWN_PAYMENT) AS prev_avg_down_payment_rate
    FROM previous_application
    GROUP BY SK_ID_CURR
),

pos_agg AS (
    SELECT SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS pos_n_loans,
        COUNT(*) AS pos_n_months,
        MAX(SK_DPD) AS pos_max_dpd,
        AVG(SK_DPD > 0) AS pos_pct_months_dpd,
        AVG(CNT_INSTALMENT_FUTURE) AS pos_avg_instalments_left
    FROM pos_cash_balance
    GROUP BY SK_ID_CURR
),

cc_agg AS (
    SELECT SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS cc_n_cards,
        COUNT(*) AS cc_n_months,
        AVG(AMT_BALANCE) AS cc_avg_balance,
        MAX(AMT_BALANCE) AS cc_max_balance,
        AVG(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0)) AS cc_avg_utilization,
        MAX(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0)) AS cc_max_utilization,
        AVG(AMT_DRAWINGS_CURRENT) AS cc_avg_drawings,
        MAX(SK_DPD) AS cc_max_dpd,
        AVG(SK_DPD > 0) AS cc_pct_months_dpd
    FROM credit_card_balance
    GROUP BY SK_ID_CURR
),

inst AS (
    -- one row per installment: partial payments are added back together
    SELECT SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER,
        MAX(AMT_INSTALMENT) AS due_amt,
        SUM(AMT_PAYMENT) AS paid_amt,
        MIN(DAYS_INSTALMENT) AS due_day,
        MAX(DAYS_ENTRY_PAYMENT) AS paid_day
    FROM installments_payments
    GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER
),

inst_agg AS (
    SELECT SK_ID_CURR,
        COUNT(*) AS inst_n_payments,
        AVG(paid_day > due_day) AS inst_pct_late,
        AVG(MAX(paid_day - due_day, 0)) AS inst_avg_days_late,
        MAX(paid_day - due_day) AS inst_max_days_late,
        AVG(paid_amt < due_amt - 0.01) AS inst_pct_underpaid,
        SUM(MAX(due_amt - paid_amt, 0)) AS inst_total_shortfall,
        AVG(paid_amt / NULLIF(due_amt, 0)) AS inst_avg_paid_ratio
    FROM inst
    GROUP BY SK_ID_CURR
)

SELECT *
FROM (SELECT SK_ID_CURR FROM application) AS a
LEFT JOIN bureau_agg USING (SK_ID_CURR)
LEFT JOIN prev_agg USING (SK_ID_CURR)
LEFT JOIN pos_agg USING (SK_ID_CURR)
LEFT JOIN cc_agg USING (SK_ID_CURR)
LEFT JOIN inst_agg USING (SK_ID_CURR)
"""

history = run_query("history summaries", HISTORY_SQL)


history summaries: 356,255 rows x 47 columns in 107s


# Applicant Payament Change
**Purpose of this cell**: This will allow us to create signal for the model to pick up on when there is a change of payament behavior, and whether it is gettting better or worse. So the cell will look at a client who paid well for 4 years and if they happen to have a slip up all of a sudden sometime like spring time, it will still mark them down as a reliable applicant who pays on time. The cell will produce a signal if a long timeapplicant that had good standing history or bad suddenly changes for a long period up till now. This way it ensures that our model is able to capture the change within a applicants history to improve the model's ability to classify applicants who will and will not struggle paying on time. 

In [6]:
INST_CHANGE_SQL = """
WITH inst AS (
    -- one row per installment: partial payments are added back together
    SELECT SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER,
        MAX(AMT_INSTALMENT) AS due_amt,
        SUM(AMT_PAYMENT) AS paid_amt,
        MIN(DAYS_INSTALMENT) AS due_day,
        MAX(DAYS_ENTRY_PAYMENT) AS paid_day
    FROM installments_payments
    GROUP BY SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_NUMBER
),

p AS (
    SELECT SK_ID_CURR,
        due_day AS DAYS_INSTALMENT,
        (paid_day > due_day) AS is_late,
        MAX(paid_day - due_day, 0) AS days_late,
        (paid_amt < due_amt - 0.01) AS is_underpaid,
        paid_amt AS AMT_PAYMENT
    FROM inst
)

SELECT SK_ID_CURR,
    SUM(DAYS_INSTALMENT >= -180) AS inst_n_6m,
    SUM(DAYS_INSTALMENT >= -365) AS inst_n_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN is_late END) AS inst_pct_late_6m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN is_late END) AS inst_pct_late_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN days_late END) AS inst_avg_days_late_6m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN days_late END) AS inst_avg_days_late_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN is_underpaid END) AS inst_pct_underpaid_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN is_late END)
        - AVG(CASE WHEN DAYS_INSTALMENT < -180 THEN is_late END) AS inst_late_trend_6m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN is_late END)
        - AVG(CASE WHEN DAYS_INSTALMENT < -365 THEN is_late END) AS inst_late_trend_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN days_late END)
        - AVG(CASE WHEN DAYS_INSTALMENT < -365 THEN days_late END) AS inst_days_late_trend_12m,
    MAX(CASE WHEN is_late = 1 THEN DAYS_INSTALMENT END) AS inst_last_late_days,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN AMT_PAYMENT END)
        / NULLIF(AVG(CASE WHEN DAYS_INSTALMENT < -365 THEN AMT_PAYMENT END), 0) AS inst_payment_change_ratio
FROM p
GROUP BY SK_ID_CURR

"""
inst_change = run_query("installment change", INST_CHANGE_SQL)
inst_change.describe().T


installment change: 339,587 rows x 13 columns in 55s


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,339587.0,278154.892278,102880.492598,100001.000000,189042.500000,278238.000000,367315.500000,456255.000000
inst_n_6m,339587.0,4.665405,5.376765,0.000000,0.000000,4.000000,6.000000,120.000000
inst_n_12m,339587.0,9.643850,9.985809,0.000000,0.000000,8.000000,13.000000,174.000000
inst_pct_late_6m,211207.0,0.059358,0.152783,0.000000,0.000000,0.000000,0.000000,1.000000
inst_pct_late_12m,252727.0,0.063916,0.149582,0.000000,0.000000,0.000000,0.050000,1.000000
inst_avg_days_late_6m,211207.0,0.241009,1.297440,0.000000,0.000000,0.000000,0.000000,115.000000
inst_avg_days_late_12m,252727.0,0.307917,1.654035,0.000000,0.000000,0.000000,0.086957,308.000000
inst_pct_underpaid_12m,252727.0,0.000638,0.011565,0.000000,0.000000,0.000000,0.000000,1.000000
inst_late_trend_6m,206227.0,-0.015106,0.164243,-1.000000,-0.058824,0.000000,0.000000,1.000000
inst_late_trend_12m,223967.0,-0.016203,0.172372,-1.000000,-0.058824,0.000000,0.000000,1.000000


# Credit Card Changes Features


**PURPOSE of this cell** Financial strain building up on credit cards before the application. This sql command is looking into two things: whether an applicant is running at the limit and paying only the minimum.

In [7]:
CC_CHANGE_SQL = """
WITH c AS (
    SELECT SK_ID_CURR,
        MONTHS_BALANCE,
        AMT_BALANCE,
        AMT_DRAWINGS_CURRENT,
        SK_DPD,
        AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0) AS util,
        CASE WHEN AMT_INST_MIN_REGULARITY > 0
             THEN (AMT_PAYMENT_TOTAL_CURRENT <= AMT_INST_MIN_REGULARITY) END AS paid_min_only
    FROM credit_card_balance
)
SELECT SK_ID_CURR,
    SUM(MONTHS_BALANCE >= -12) AS cc_n_months_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN util END) AS cc_util_3m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN util END) AS cc_util_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN util END)
        - AVG(CASE WHEN MONTHS_BALANCE < -3 THEN util END) AS cc_util_trend_3m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN util >= 0.9 END) AS cc_pct_maxed_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN AMT_BALANCE END)
        - AVG(CASE WHEN MONTHS_BALANCE BETWEEN -12 AND -4 THEN AMT_BALANCE END) AS cc_balance_change_3m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN AMT_DRAWINGS_CURRENT END)
        / NULLIF(AVG(CASE WHEN MONTHS_BALANCE < -3 THEN AMT_DRAWINGS_CURRENT END), 0) AS cc_drawings_change_ratio,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN paid_min_only END) AS cc_pct_min_payment_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN SK_DPD > 0 END) AS cc_pct_dpd_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN SK_DPD > 0 END)
        - AVG(CASE WHEN MONTHS_BALANCE < -12 THEN SK_DPD > 0 END) AS cc_dpd_trend_12m
FROM c
GROUP BY SK_ID_CURR
"""

cc_change = run_query("Credit Card Change", CC_CHANGE_SQL)
cc_change.describe().T

Credit Card Change: 103,558 rows x 11 columns in 6s


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,103558.0,278381.457956,102779.519683,100006.000000,189536.250000,278649.000000,367690.000000,456250.000000
cc_n_months_12m,103558.0,10.311671,2.558131,1.000000,10.000000,11.000000,12.000000,36.000000
cc_util_3m,77751.0,0.347040,0.428874,0.000000,0.000000,0.000000,0.854494,8.100310
cc_util_12m,84881.0,0.294425,0.369034,0.000000,0.000000,0.010389,0.631812,2.708565
cc_util_trend_3m,75823.0,0.024700,0.288997,-1.338251,-0.053083,0.000000,0.151208,7.213436
cc_pct_maxed_12m,84881.0,0.190326,0.316275,0.000000,0.000000,0.000000,0.333333,1.000000
cc_balance_change_3m,98576.0,11110.461202,71757.674448,-916486.507500,0.000000,0.000000,2929.982500,844769.103000
cc_drawings_change_ratio,67810.0,0.639762,11.645379,0.000000,0.000000,0.000000,0.330656,2880.260526
cc_pct_min_payment_12m,44255.0,0.265454,0.316153,0.000000,0.000000,0.142857,0.500000,1.000000
cc_pct_dpd_12m,103558.0,0.012250,0.090963,0.000000,0.000000,0.000000,0.000000,1.000000


# Curent loans vs previous borrowing, and recent credit seeking

The purpose of this data frame is going to look at two things within applicants:

1. Whether the loan being requested now is ussually large compared with what the applicant has previously borrowed
2. Whether they've been applying for and taking out credit more often....

In [8]:
BORROW_CHANGE_SQL = """
WITH prev AS (
    SELECT SK_ID_CURR,
        SUM(DAYS_DECISION >= -365) AS prev_n_apps_12m,
        SUM(DAYS_DECISION >= -365 AND NAME_CONTRACT_STATUS = 'Refused') AS prev_n_refused_12m,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' AND AMT_CREDIT > 0 THEN AMT_CREDIT END) AS prev_avg_approved_credit,
        MAX(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN AMT_CREDIT END) AS prev_max_approved_credit,
        AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' AND AMT_ANNUITY > 0 THEN AMT_ANNUITY END) AS prev_avg_approved_annuity,
        AVG(CASE WHEN DAYS_DECISION >= -730 THEN AMT_APPLICATION END)
            / NULLIF(AVG(CASE WHEN DAYS_DECISION < -730 THEN AMT_APPLICATION END), 0) AS prev_applied_change_ratio
    FROM previous_application
    GROUP BY SK_ID_CURR
),

bur AS (
    SELECT SK_ID_CURR,
        SUM(DAYS_CREDIT >= -365) AS bureau_n_new_12m,
        SUM(DAYS_CREDIT >= -730) AS bureau_n_new_24m,
        SUM(CASE WHEN DAYS_CREDIT >= -730 THEN AMT_CREDIT_SUM_DEBT END) AS bureau_new_debt_24m
    FROM bureau
    GROUP BY SK_ID_CURR
)

SELECT a.SK_ID_CURR,
    p.prev_n_apps_12m,
    p.prev_n_refused_12m,
    p.prev_applied_change_ratio,
    a.AMT_CREDIT / NULLIF(p.prev_avg_approved_credit, 0) AS cur_credit_vs_prev_avg,
    a.AMT_CREDIT / NULLIF(p.prev_max_approved_credit, 0) AS cur_credit_vs_prev_max,
    a.AMT_ANNUITY / NULLIF(p.prev_avg_approved_annuity, 0) AS cur_annuity_vs_prev_avg,
    b.bureau_n_new_12m,
    b.bureau_n_new_24m,
    b.bureau_new_debt_24m
FROM application AS a
LEFT JOIN prev AS p ON a.SK_ID_CURR = p.SK_ID_CURR
LEFT JOIN bur AS b ON a.SK_ID_CURR = b.SK_ID_CURR
"""

borrow_change = run_query("borrow change", BORROW_CHANGE_SQL)
borrow_change.describe().T

borrow change: 356,255 rows x 10 columns in 9s


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,356255.0,278128.000000,1.028421e+05,1.000010e+05,189064.500000,278128.000000,367191.500000,4.562550e+05
prev_n_apps_12m,338857.0,1.693163,2.536145e+00,0.000000e+00,0.000000,1.000000,2.000000,6.300000e+01
prev_n_refused_12m,338857.0,0.338535,1.086351e+00,0.000000e+00,0.000000,0.000000,0.000000,5.400000e+01
prev_applied_change_ratio,189380.0,2.723635,4.409413e+00,0.000000e+00,0.664733,1.437916,3.059082,2.263980e+02
cur_credit_vs_prev_avg,337689.0,5.834692,7.066325e+00,5.582540e-02,1.793429,3.450799,7.080350,1.492537e+02
cur_credit_vs_prev_max,337689.0,4.491852,6.543526e+00,2.132000e-02,1.000000,2.204004,5.293970,1.492537e+02
cur_annuity_vs_prev_avg,337651.0,2.676296,2.249195e+00,8.470970e-02,1.280717,2.040130,3.323588,7.217874e+01
bureau_n_new_12m,305811.0,1.059664,1.421335e+00,0.000000e+00,0.000000,1.000000,2.000000,8.000000e+01
bureau_n_new_24m,305811.0,2.152153,2.185761e+00,0.000000e+00,1.000000,2.000000,3.000000,9.600000e+01
bureau_new_debt_24m,230499.0,625829.044756,1.427181e+06,-2.089184e+06,21142.845000,185582.025000,651226.500000,6.457024e+07


# Affordability and thin-file features......

This code is going to be used to describe each applicant. It will create a feature that will highlight their **financial capaicty** and how much **credit history** they have. These features that measure money and history directly (from the data) so that the model can judge applicants on their finances rather than on their age (**TO REDUCE AGE BIAS**). 

In [9]:
cap = df_base[["SK_ID_CURR", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "CNT_FAM_MEMBERS", "DAYS_BIRTH", "DAYS_EMPLOYED",
               "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].merge(
    history[["SK_ID_CURR", "bureau_n_loans", "prev_n_apps", "bureau_oldest_loan_days", "bureau_total_debt"]],
    on="SK_ID_CURR", how="left",
)

In [10]:
family = cap["CNT_FAM_MEMBERS"].replace(0, np.nan) #number of family members

cap["CREDIT_INCOME_RATIO"] = cap["AMT_CREDIT"] / cap["AMT_INCOME_TOTAL"]
cap["ANNUITY_INCOME_RATIO"] = cap["AMT_ANNUITY"] / cap["AMT_INCOME_TOTAL"]
cap["CREDIT_TERM"] = cap["AMT_ANNUITY"] / cap["AMT_CREDIT"]
cap["GOODS_CREDIT_RATIO"] = cap["AMT_GOODS_PRICE"] / cap["AMT_CREDIT"]
cap["AGE_YEARS"] = -cap["DAYS_BIRTH"] / 365.25 # going to use exact number
cap["EMPLOYED_AGE_RATIO"] = cap["DAYS_EMPLOYED"] / cap["DAYS_BIRTH"]

cap["INCOME_PER_PERSON"] = cap["AMT_INCOME_TOTAL"] / family # income divided by the amount of family members
cap["INCOME_LEFT_PER_PERSON"] = (cap["AMT_INCOME_TOTAL"] - cap["AMT_ANNUITY"]) / family # NOTE Anuity is a set payment or how much is expected to be paid 
# essentially this code will allow us to see how much a person is making subtract the amount they are expected to pay
cap["TOTAL_DEBT_TO_INCOME"] = (cap["bureau_total_debt"].fillna(0) + cap["AMT_CREDIT"]) / cap["AMT_INCOME_TOTAL"]

cap["N_PRIOR_CREDITS"] = cap["bureau_n_loans"].fillna(0) + cap["prev_n_apps"].fillna(0) # tracks how many previous applications and loans an applicant has
cap["THIN_FILE"] = (cap["N_PRIOR_CREDITS"] == 0).astype(int)
cap["CREDIT_HISTORY_YEARS"] = -cap["bureau_oldest_loan_days"] / 365.25


ext = cap[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]] # the three outside credit scores
cap["EXT_MEAN"] = ext.mean(axis=1)  # average of whichever scores are present
cap["EXT_MIN"] = ext.min(axis=1)  # weakest score
cap["EXT_MAX"] = ext.max(axis=1)  # strongest score
cap["EXT_STD"] = ext.std(axis=1)  # how much the agencies disagree (NaN if only one score)
cap["EXT_PROD"] = ext.prod(axis=1, min_count=3)  # only when all three are present, otherwise NaN
cap["EXT_N_MISSING"] = ext.isna().sum(axis=1)  # 0 to 3 missing scores
cap["EXT_MEAN_X_CREDIT_TERM"] = cap["EXT_MEAN"] * cap["CREDIT_TERM"]  # score combined with how fast the loan is repaid


new_cap_cols = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM", "GOODS_CREDIT_RATIO",
                "AGE_YEARS", "EMPLOYED_AGE_RATIO", "INCOME_PER_PERSON", "INCOME_LEFT_PER_PERSON",
                "TOTAL_DEBT_TO_INCOME", "N_PRIOR_CREDITS", "THIN_FILE", "CREDIT_HISTORY_YEARS",
                "EXT_MEAN", "EXT_MIN", "EXT_MAX", "EXT_STD", "EXT_PROD", "EXT_N_MISSING", "EXT_MEAN_X_CREDIT_TERM"]
afford = cap[["SK_ID_CURR"] + new_cap_cols] # new afford table

print(afford.shape)
afford[new_cap_cols].describe().T


(356255, 20)


,count,mean,std,min,25%,50%,75%,max
CREDIT_INCOME_RATIO,356255.0,3.849476,2.635035,4.807615e-03,2.000000,3.158857,5.000000,8.473684e+01
ANNUITY_INCOME_RATIO,356219.0,0.181211,0.094701,2.238846e-04,0.114950,0.163182,0.229156,2.024714e+00
CREDIT_TERM,356219.0,0.055463,0.023206,2.207258e-02,0.038080,0.050000,0.067066,1.244296e-01
GOODS_CREDIT_RATIO,355977.0,0.899716,0.096140,1.666667e-01,0.834725,0.893815,1.000000,6.666667e+00
AGE_YEARS,356255.0,43.918546,11.933755,2.009035e+01,34.017796,43.134839,53.869952,6.907324e+01
EMPLOYED_AGE_RATIO,291607.0,0.157798,0.133450,-0.000000e+00,0.056954,0.120075,0.220598,7.288115e-01
INCOME_PER_PERSON,356253.0,93786.296339,98100.856971,2.812500e+03,49500.000000,75000.000000,112500.000000,3.900000e+07
INCOME_LEFT_PER_PERSON,356217.0,78899.155154,93844.298677,-1.613925e+04,39080.250000,61746.750000,98349.750000,3.899127e+07
TOTAL_DEBT_TO_INCOME,356255.0,6.967157,8.007766,-1.276457e+01,2.976402,5.060444,8.458118,1.656510e+03
N_PRIOR_CREDITS,356255.0,9.506230,6.810209,0.000000e+00,5.000000,8.000000,13.000000,1.220000e+02


## Joining everything and creating a filw to store the new tables...


In [11]:
full = df_base
for name, part in [("history", history), ("inst_change", inst_change), ("cc_change", cc_change), ("borrow_change", borrow_change), ("afford", afford)]:
    full = full.merge(part, on="SK_ID_CURR", how="left", validate="one_to_one")
    print(f"after {name} {full.shape}")

after history (356255, 170)
after inst_change (356255, 182)
after cc_change (356255, 192)
after borrow_change (356255, 201)
after afford (356255, 220)


In [12]:
assert full["SK_ID_CURR"].is_unique
assert not full.columns.duplicated().any()
assert len(full) == len(df_base)

In [13]:
train_fe = full[full["dataset"] == "train"].reset_index(drop=True)
test_fe = full[full["dataset"] == "test"].reset_index(drop=True)

train_fe.to_parquet(PROCESSED / "train_fe.parquet", index=False)
test_fe.to_parquet(PROCESSED / "test_fe.parquet", index=False)

new_feature_cols = [c for part in (inst_change, cc_change, borrow_change, afford)
                    for c in part.columns if c != "SK_ID_CURR"]

print(f"train_fe: {train_fe.shape}   test_fe: {test_fe.shape}")
print(f"{len(new_feature_cols)} engineered columns (features) added in this notebook")

train_fe: (307511, 220)   test_fe: (48744, 220)
50 engineered columns (features) added in this notebook


# CHECK **MOST IMPORTANT**
This is impoarantto construct because we conducting a lot of feature engineering. Lots of our data had work done on it and each individual table we had created encoporated a articulate operation to allow our model to pick up signal produced within our data.


1. Checks how much is missing
2. any infinite values, which would come from bad division
3. how well the feature separates defaulters on its own, as a single feature AUC

**IMPORTANT** this is a test to see if we can capture whether clients whose behavior got worse recently default more often.

In [14]:
from sklearn.metrics import roc_auc_score # using this to check weather or not the data is present or not... 

y_fe = train_fe["TARGET"].astype(int)

checks = []
for col in new_feature_cols:
    s = train_fe[col]  # checking every col
    known = s.notna()
    auc = roc_auc_score(y_fe[known], s[known]) if s[known].nunique() > 1 else np.nan # checks to see if there are more than one unknown then lets check
    checks.append({
        "feature": col,
        "pct_missing": 1 - known.mean(),
        "n_inf": int(np.isinf(s).sum()),
        "auc": max(auc, 1 - auc),
        "direction": "higher = riskier" if auc > 0.5 else "lower = riskier",
        "default_if_missing": y_fe[~known].mean() if (~known).any() else np.nan,
    }) # giving us a list of how much is missing

checks = pd.DataFrame(checks).set_index("feature").sort_values("auc", ascending=False)
print(checks.round(3))

def trend_table(col):
    """
    This code will label whether or not the trend of applicants are getting worse or not.
    """
    band = pd.cut(train_fe[col], [-np.inf, -0.05, 0.05, np.inf], labels=["improving", "steady", "worsening"])
    band = band.cat.add_categories("no recent data").fillna("no recent data")
    return train_fe.groupby(band, observed=True)["TARGET"].agg(clients="size", default_rate="mean")

for col in ["inst_late_trend_6m", "inst_late_trend_12m", "cc_util_trend_3m", "cc_dpd_trend_12m"]:
    print(f"\n--- {col} ---")
    print(trend_table(col).round(3))

print("\n--- THIN_FILE ---")
print(train_fe.groupby("THIN_FILE")["TARGET"].agg(clients="size", default_rate="mean").round(3))

                           pct_missing  n_inf    auc         direction  \
feature                                                                  
EXT_PROD                         0.644      0  0.740   lower = riskier   
EXT_MEAN                         0.001      0  0.717   lower = riskier   
EXT_MIN                          0.001      0  0.691   lower = riskier   
EXT_MAX                          0.001      0  0.686   lower = riskier   
cc_util_3m                       0.787      0  0.650  higher = riskier   
cc_pct_maxed_12m                 0.768      0  0.640  higher = riskier   
cc_util_12m                      0.768      0  0.639  higher = riskier   
EXT_MEAN_X_CREDIT_TERM           0.001      0  0.638   lower = riskier   
cc_util_trend_3m                 0.793      0  0.587  higher = riskier   
AGE_YEARS                        0.000      0  0.583   lower = riskier   
bureau_n_new_12m                 0.143      0  0.580  higher = riskier   
inst_last_late_days              0.496

# Summarizing our findings.... (View the readme document Feature engineering project)
The **core idea** is to reflect that the worse the feature or for each individual getting worse within each column in the data, it shows that there is a result of more defaulters.

| Label | Rule | Meaning |
|---|---|---|
| **Improving** | $\text{trend} < -0.05$ | Recent rate is more than 5 points **better** than before the window |
| **Steady** | $-0.05 \le \text{trend} \le 0.05$ | Within 5 points of before the window |
| **Worsening** | $\text{trend} > 0.05$ | Recent rate is more than 5 points **worse** than before the window |
| **No recent data** | trend is `NaN` | No records in the recent window, or none before it |

# Model TEST with New features
## First test LightGBM model...

In [15]:
import warnings
import duckdb   
import lightgbm as lgb
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore", message="The argument 'eval_set' is deprecated")

s = duckdb.sql("SELECT SK_ID_CURR, split FROM 'data_splits/stratified_target.parquet'").df()
split = train_fe["SK_ID_CURR"].map(s.set_index("SK_ID_CURR")["split"])
tr, dv, va = [(split == k).to_numpy() for k in ["train", "dev", "validation"]]

# gotta get rid of our old summarized data
OLD_RATIOS = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM", "GOODS_CREDIT_RATIO", "AGE_YEARS", "EMPLOYED_AGE_RATIO"]

all_features = [c for c in train_fe.columns if c not in ("SK_ID_CURR", "dataset", "TARGET")]
truly_new = [c for c in new_feature_cols  if c not in OLD_RATIOS] # extracting only the new features
baseline_features = [c for c in all_features if c not in truly_new] # extracting allfeatures and combining the column names...
print(f"baseline: {len(baseline_features)}, with new: {len(all_features)}, new features {len(truly_new)}")

x_all = train_fe[all_features].copy()
y_all = train_fe["TARGET"].astype(int) # grabs and ensure the target is an integer... also making our target variables
cat_cols = x_all.select_dtypes(include=["object", "str"]).columns.tolist() # grabbing the columns for object / str. important for NN 
x_all[cat_cols] = x_all[cat_cols].fillna("MISSING") # similar to our previous model we are adding missing to categorical columns
for col in cat_cols: 
    # this is ensuring that our categorical columns are sound and are working
    cats = pd.Categorical(x_all.loc[tr | dv, col]).categories
    x_all[col] = pd.Categorical(x_all[col], categories=cats)
    
cv_rows = np.where(tr | dv)[0]  # row positions of train + dev; validation is never used here
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=420)
folds = list(skf.split(cv_rows, y_all.iloc[cv_rows]))  # same 5 folds for every model, so comparisons are fair


def fit_lgb(features, seed):
    X = x_all[features]
    oof = np.zeros(len(cv_rows))  # out-of-fold prediction for every train + dev row
    p_va = np.zeros(va.sum())  # validation prediction, averaged over the 5 fold models
    trees = []

    for fit_idx, stop_idx in folds:
        fit_rows, stop_rows = cv_rows[fit_idx], cv_rows[stop_idx]
        model = lgb.LGBMClassifier(
            n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=100, subsample=0.8,
            subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, random_state=seed, n_jobs=-1,
            verbose=-1
        )
        model.fit(X.iloc[fit_rows], y_all.iloc[fit_rows],
                  eval_set=[(X.iloc[stop_rows], y_all.iloc[stop_rows])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(100, verbose=False)])

        oof[stop_idx] = model.predict_proba(X.iloc[stop_rows])[:, 1]
        p_va += model.predict_proba(X[va])[:, 1] / len(folds)
        trees.append(model.best_iteration_)

    scores = {
        "oof_auc": roc_auc_score(y_all.iloc[cv_rows], oof),
        "val_auc": roc_auc_score(y_all[va], p_va),
        "val_pr_auc": average_precision_score(y_all[va], p_va),
        "trees": int(np.mean(trees)),
    }
    return model, p_va, scores


baseline: 173, with new: 217, new features 44


In [16]:
SEEDS = [420]
rows, models, val_preds = [], {}, {}
for version, feats in [("baseline", baseline_features), ("with new features", all_features)]:
    for seed in SEEDS:
        model, p_va, scores = fit_lgb(feats, seed)
        models[(version, seed)] = model
        val_preds[(version, seed)] = p_va
        rows.append({"version": version, "seed": seed, **scores})
        print(f"{version:<18} seed {seed:<5} OOF AUC {scores['oof_auc']:.4f}   val AUC {scores['val_auc']:.4f}")


results_fe = pd.DataFrame(rows)
summary_fe = results_fe.groupby("version")[["oof_auc", "val_auc", "val_pr_auc"]].agg(["mean", "std"]).round(4)
print(summary_fe)

gain = summary_fe.loc["with new features", ("val_auc", "mean")] - summary_fe.loc["baseline", ("val_auc", "mean")]
print(f"\nValidation AUC gain from new features: {gain:+.4f}")

baseline           seed 420   OOF AUC 0.7859   val AUC 0.7887
with new features  seed 420   OOF AUC 0.7888   val AUC 0.7912
                  oof_auc     val_auc     val_pr_auc    
                     mean std    mean std       mean std
version                                                 
baseline           0.7859 NaN  0.7887 NaN     0.2758 NaN
with new features  0.7888 NaN  0.7912 NaN     0.2791 NaN

Validation AUC gain from new features: +0.0025


In [17]:
from itertools import product

BASE_PARAMS = dict(n_estimators=6000, learning_rate=0.03, subsample=0.8, subsample_freq=1,
                   n_jobs=-1, verbose=-1)

def fit_tuned(features, params, seed):
    model = lgb.LGBMClassifier(**BASE_PARAMS, **params, random_state=seed)
    model.fit(x_all.loc[tr, features], y_all[tr],
              eval_set=[(x_all.loc[dv, features], y_all[dv])], eval_metric="auc",
              callbacks=[lgb.early_stopping(100, verbose=False)])
    p_dv = model.predict_proba(x_all.loc[dv, features])[:, 1]
    p_va = model.predict_proba(x_all.loc[va, features])[:, 1]
    return model, p_dv, p_va

grid = {
    "num_leaves": [4, 5, 7],
    "min_child_samples": [400, 800, 1500],
    "colsample_bytree": [0.3, 0.4],
    "reg_lambda": [1, 10],
    "reg_alpha": [1],
}
n_combos = int(np.prod([len(v) for v in grid.values()]))

tune_rows = []
for values in product(*grid.values()):
    params = dict(zip(grid.keys(), values))
    model, p_dv, _ = fit_tuned(all_features, params, seed=420)
    dev_auc = roc_auc_score(y_all[dv], p_dv)
    tune_rows.append({**params, "dev_auc": dev_auc, "trees": model.best_iteration_})
    print(f"{len(tune_rows):>2}/{n_combos}  {params}  dev AUC {dev_auc:.4f}  trees {model.best_iteration_}")

tune_fe = pd.DataFrame(tune_rows).sort_values("dev_auc", ascending=False).reset_index(drop=True)
print("\nTop 10 settings by dev AUC:")
print(tune_fe.head(10).round(4))

TOP_K = 5
ENS_SEEDS = [1, 2, 3, 4]
top_params = tune_fe.head(TOP_K)[list(grid.keys())].to_dict("records")

member_dv, member_va, member_val_aucs, tuned_models = [], [], [], []
for k, params in enumerate(top_params, start=1):
    for seed in ENS_SEEDS:
        model, p_dv, p_va = fit_tuned(all_features, params, seed)
        tuned_models.append(model)
        member_dv.append(p_dv)
        member_va.append(p_va)
        member_val_aucs.append(roc_auc_score(y_all[va], p_va))
        print(f"ensemble member: config {k}/{TOP_K}, seed {seed}  val AUC {member_val_aucs[-1]:.4f}")

p_tuned_dv = np.mean(member_dv, axis=0)
p_tuned_va = np.mean(member_va, axis=0)
val_preds[("tuned", "ensemble")] = p_tuned_va

final_compare = pd.DataFrame({
    "default settings, 5-fold CV": {
        "val_auc": roc_auc_score(y_all[va], val_preds[("with new features", SEEDS[0])]),
        "val_pr_auc": average_precision_score(y_all[va], val_preds[("with new features", SEEDS[0])]),
    },
    "tuned, best single model": {
        "val_auc": max(member_val_aucs),
        "val_pr_auc": np.nan,
    },
    f"tuned, {len(tuned_models)}-model ensemble": {
        "val_auc": roc_auc_score(y_all[va], p_tuned_va),
        "val_pr_auc": average_precision_score(y_all[va], p_tuned_va),
    },
}).T.round(4)

print(f"\nTuned ensemble dev AUC: {roc_auc_score(y_all[dv], p_tuned_dv):.4f}")
final_compare


 1/36  {'num_leaves': 4, 'min_child_samples': 400, 'colsample_bytree': 0.3, 'reg_lambda': 1, 'reg_alpha': 1}  dev AUC 0.7906  trees 3908
 2/36  {'num_leaves': 4, 'min_child_samples': 400, 'colsample_bytree': 0.3, 'reg_lambda': 10, 'reg_alpha': 1}  dev AUC 0.7908  trees 4575
 3/36  {'num_leaves': 4, 'min_child_samples': 400, 'colsample_bytree': 0.4, 'reg_lambda': 1, 'reg_alpha': 1}  dev AUC 0.7901  trees 3600
 4/36  {'num_leaves': 4, 'min_child_samples': 400, 'colsample_bytree': 0.4, 'reg_lambda': 10, 'reg_alpha': 1}  dev AUC 0.7901  trees 4304
 5/36  {'num_leaves': 4, 'min_child_samples': 800, 'colsample_bytree': 0.3, 'reg_lambda': 1, 'reg_alpha': 1}  dev AUC 0.7911  trees 4461
 6/36  {'num_leaves': 4, 'min_child_samples': 800, 'colsample_bytree': 0.3, 'reg_lambda': 10, 'reg_alpha': 1}  dev AUC 0.7907  trees 4489
 7/36  {'num_leaves': 4, 'min_child_samples': 800, 'colsample_bytree': 0.4, 'reg_lambda': 1, 'reg_alpha': 1}  dev AUC 0.7898  trees 3085
 8/36  {'num_leaves': 4, 'min_child_sa

,val_auc,val_pr_auc
"default settings, 5-fold CV",0.7912,0.2791
"tuned, best single model",0.7914,NaN
"tuned, 20-model ensemble",0.7915,0.2787
